<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Qwen3-TTS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Qwen3-TTS — Custom Voice, Voice Design, and Voice Cloning

State-of-the-art open-source TTS by the Qwen team at Alibaba Cloud. One notebook, three modes: pick from 9 premium voices, design a brand-new voice from a text description, or clone any voice from a 3-second reference clip. 10 languages, Apache 2.0.

### Quick Start
1. Connect to a GPU runtime (A100, L4, or T4 — 16 GB+ VRAM)
2. Run Steps 1–4 in order — no token, no sign-up needed
3. Open the Gradio UI link from Step 4 and start generating

| GPU | VRAM | Model | Mode |
|-----|------|-------|------|
|-----|------|-------|------|
|-----|------|-------|------|
| A100 | 40 GB | 1.7B (recommended) | Standard |
|-----|------|-------|------|
|-----|------|-------|------|
|-----|------|-------|------|
| L4 | 24 GB | 1.7B | Standard |
|-----|------|-------|------|
|-----|------|-------|------|
|-----|------|-------|------|
| T4 | 16 GB | 0.6B | Low-VRAM |

The notebook auto-detects your GPU and picks the right model size. All three modes (Custom Voice, Voice Design, Voice Clone) are available in the Gradio UI.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the `qwen-tts` package, FlashAttention 2, and supporting libraries.

import os, sys, subprocess, importlib

# Verify GPU is available
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime → Change runtime type → T4 / L4 / A100')

gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
arch = f"{cap[0]}{cap[1]}"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB — sm_{arch}')

# Pick model size based on VRAM
MODEL_SIZE = '0.6B' if vram_gb < 20 else '1.7B'
print(f'[Model] Auto-selected {MODEL_SIZE} variant ({vram_gb:.0f} GB VRAM available)')

# Install qwen-tts and audio tools
print('[pip] Installing qwen-tts ...')
subprocess.run(['pip', 'install', '-q', '-U', 'qwen-tts'], check=True)

# FlashAttention 2 (huge VRAM savings for the 1.7B model on T4/L4)
print('[pip] Installing FlashAttention 2 ...')
try:
    subprocess.run(['pip', 'install', '-q', '-U', 'flash-attn', '--no-build-isolation'], check=True)
    print('  flash-attn — ready')
except Exception as e:
    print(f'  flash-attn failed ({e}); will fall back to sdpa attention')
    FLASH_OK = False
else:
    FLASH_OK = True

# Audio playback + Gradio
subprocess.run(['pip', 'install', '-q', 'gradio', 'soundfile', 'numpy', 'librosa'], check=True)
print('[pip] Done.')

# Output dir
os.makedirs('/content/audio_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/audio_out and /content/ref_audio ready.')

In [ ]:
# @title STEP 2 — Pre-cache Models
# @markdown Downloads the Qwen3-TTS model weights to the local Colab cache. Skip if you want them to download on first use.

# @markdown Reuses the Drive-cached weights from `TTS_Model_Loader.ipynb` if present,
# @markdown otherwise downloads to the default ~/.cache/huggingface (in-session only).
import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
    print(f'[Cache] using Drive at {CACHE_ROOT}')
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface')


import os
from huggingface_hub import snapshot_download

MODELS = {
    'CustomVoice': f'Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-CustomVoice',
    'VoiceDesign': f'Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-VoiceDesign' if MODEL_SIZE == '1.7B' else None,
    'Base':        f'Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base',
}
MODELS = {k: v for k, v in MODELS.items() if v}
TOKENIZER = 'Qwen/Qwen3-TTS-Tokenizer-12Hz'

print('[Download] Tokenizer + models (≈5 GB for 0.6B, ≈11 GB for 1.7B)...')
snapshot_download(TOKENIZER)
for tag, repo in MODELS.items():
    print(f'  → {repo}')
    snapshot_download(repo)
print('[Download] All weights cached.')

In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Defines the inference wrappers. Models are loaded on first use, not at import time, so this cell is fast.

import os, time, gc
import numpy as np
import torch
import soundfile as sf

from qwen_tts import Qwen3TTSModel

CACHE = {}

def get_attn_impl():
    try:
        import flash_attn
        return 'flash_attention_2'
    except ImportError:
        return 'sdpa'

def load_model(task: str):
    """Lazy-load a Qwen3-TTS model for a given task.
    task: 'CustomVoice' | 'VoiceDesign' | 'Base'
    """
    if task in CACHE and CACHE[task] is not None:
        return CACHE[task]
    repo = {
        'CustomVoice': f'Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-CustomVoice',
        'VoiceDesign': f'Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-VoiceDesign',
        'Base':        f'Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base',
    }[task]
    print(f'[Load] {repo} ({get_attn_impl()})...')
    t0 = time.time()
    m = Qwen3TTSModel.from_pretrained(
        repo,
        device_map='cuda:0',
        dtype=torch.bfloat16,
        attn_implementation=get_attn_impl(),
    )
    print(f'[Load] {repo} ready in {time.time()-t0:.1f}s')
    CACHE[task] = m
    return m

def free_model(task: str):
    if task in CACHE and CACHE[task] is not None:
        del CACHE[task]
        gc.collect()
        torch.cuda.empty_cache()
        print(f'[Free] {task} released from VRAM')

# Speaker table (CustomVoice only) — populated dynamically after first model load
SPEAKERS = []
LANGUAGES = ['Auto', 'Chinese', 'English', 'Japanese', 'Korean', 'German', 'French', 'Russian', 'Portuguese', 'Spanish', 'Italian']

def refresh_speakers():
    global SPEAKERS
    if not SPEAKERS:
        try:
            m = load_model('CustomVoice')
            SPEAKERS = list(m.get_supported_speakers())
        except Exception as e:
            print(f'[Speakers] {e}')
            SPEAKERS = ['Vivian', 'Serena', 'Uncle_Fu', 'Dylan', 'Eric', 'Ryan', 'Aiden', 'Ono_Anna', 'Sohee']
    return SPEAKERS

def save_wav(wav: np.ndarray, sr: int, name: str) -> str:
    path = os.path.join('/content/audio_out', name)
    sf.write(path, wav, sr)
    return path

def synth_custom_voice(text, language, speaker, instruct):
    m = load_model('CustomVoice')
    wavs, sr = m.generate_custom_voice(
        text=text,
        language=None if language == 'Auto' else language,
        speaker=speaker,
        instruct=instruct or None,
    )
    return save_wav(wavs[0], sr, 'custom_voice.wav'), sr

def synth_voice_design(text, language, instruct):
    if MODEL_SIZE == '0.6B':
        raise RuntimeError('Voice Design needs the 1.7B model. Reconnect to a GPU with ≥20 GB VRAM.')
    m = load_model('VoiceDesign')
    wavs, sr = m.generate_voice_design(
        text=text,
        language=None if language == 'Auto' else language,
        instruct=instruct,
    )
    return save_wav(wavs[0], sr, 'voice_design.wav'), sr

def synth_voice_clone(text, language, ref_audio_path, ref_text, x_vector_only):
    if ref_audio_path is None:
        raise RuntimeError('Upload a reference audio clip first.')
    m = load_model('Base')
    if not x_vector_only and not ref_text:
        raise RuntimeError('Provide the reference transcript for best quality, or enable x-vector-only mode.')
    wavs, sr = m.generate_voice_clone(
        text=text,
        language=None if language == 'Auto' else language,
        ref_audio=ref_audio_path,
        ref_text=ref_text,
        x_vector_only_mode=bool(x_vector_only),
    )
    return save_wav(wavs[0], sr, 'voice_clone.wav'), sr

print('[Core] Ready. Use Step 4 for the UI, Step 6 for quick test, Step 7 for batch.')

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with three tabs: Custom Voice, Voice Design, Voice Clone. Click the `.gradio.live` link to open.

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr

DEFAULT_SPEAKERS = ['Vivian', 'Serena', 'Uncle_Fu', 'Dylan', 'Eric', 'Ryan', 'Aiden', 'Ono_Anna', 'Sohee']

def _err(msg):
    return None, f'❌ {msg}'

def ui_custom_voice(text, language, speaker, instruct, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    try:
        progress(0.1, desc='Loading model...')
        progress(0.4, desc='Synthesizing...')
        path, sr = synth_custom_voice(text.strip(), language, speaker, instruct.strip())
        progress(1.0, desc='Done.')
        return path, f'✅ Generated at {sr} Hz'
    except Exception as e:
        return _err(str(e))

def ui_voice_design(text, language, instruct, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    if not instruct or not instruct.strip():
        return _err('Describe the voice you want (e.g. "warm elderly British woman").')
    try:
        progress(0.1, desc='Loading model...')
        progress(0.4, desc='Synthesizing...')
        path, sr = synth_voice_design(text.strip(), language, instruct.strip())
        progress(1.0, desc='Done.')
        return path, f'✅ Generated at {sr} Hz'
    except Exception as e:
        return _err(str(e))

def ui_voice_clone(text, language, ref_audio, ref_text, x_vector_only, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    if ref_audio is None:
        return _err('Upload a reference audio clip (3+ seconds recommended).')
    try:
        progress(0.1, desc='Loading model...')
        progress(0.3, desc='Extracting speaker features...')
        progress(0.5, desc='Synthesizing...')
        path, sr = synth_voice_clone(text.strip(), language, ref_audio, ref_text or '', x_vector_only)
        progress(1.0, desc='Done.')
        return path, f'✅ Generated at {sr} Hz'
    except Exception as e:
        return _err(str(e))

def ui_free(task):
    try:
        free_model(task)
        return f'✅ {task} model released from VRAM.'
    except Exception as e:
        return f'❌ {e}'

with gr.Blocks(title='Qwen3-TTS', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# Qwen3-TTS\n**Model:** {MODEL_SIZE} · **GPU:** {gpu_name} ({vram_gb:.0f} GB) · **FlashAttention 2:** {"on" if FLASH_OK else "off (sdpa fallback)"}')
    with gr.Tabs():
        with gr.Tab('Custom Voice'):
            gr.Markdown('**Pick from 9 premium speakers** — each has a distinct gender, age, language, and dialect. Optional `instruct` adds emotion/style control.')
            with gr.Row():
                with gr.Column():
                    cv_text = gr.Textbox(label='Text', value='Hello! This is a test of Qwen3 text-to-speech.', lines=4,
                                            info='Up to a few hundred characters works well; longer is fine but generation is slower.')
                    cv_lang = gr.Dropdown(LANGUAGES, value='Auto', label='Language',
                                             info='Auto = let the model detect from the text. Pick a specific language for tricky inputs.')
                    cv_speaker = gr.Dropdown(DEFAULT_SPEAKERS, value='Ryan', label='Speaker',
                                                info='9 premium voices covering EN/ZH/JA/KO/DE/FR. Hover for the speaker description.')
                    cv_instruct = gr.Textbox(label='Instruct (optional)', value='', placeholder='e.g. "Speak cheerfully"',
                                                info='Free-form style hint: emotion, pace, tone. Leave empty for default delivery.')
                    cv_btn = gr.Button('Generate (CustomVoice)', variant='primary')
                with gr.Column():
                    cv_audio = gr.Audio(label='Output', type='filepath')
                    cv_status = gr.Markdown('')
            cv_btn.click(ui_custom_voice, [cv_text, cv_lang, cv_speaker, cv_instruct], [cv_audio, cv_status])

        with gr.Tab('Voice Design'):
            if MODEL_SIZE == '0.6B':
                gr.Markdown('⚠️ **Voice Design needs the 1.7B model.** Reconnect to a runtime with ≥20 GB VRAM (L4 or A100).')
            else:
                gr.Markdown('**Design a brand-new voice from a natural-language description.** Be specific: age, gender, accent, emotion, pitch, pace.')
                with gr.Row():
                    with gr.Column():
                        vd_text = gr.Textbox(label='Text', value='哥哥，你回来啦，人家等了你好久好久了，要抱抱！', lines=4,
                                            info='Any text. Designed voices work best with longer sentences that show off the character.')
                        vd_lang = gr.Dropdown(LANGUAGES, value='Chinese', label='Language',
                                             info='Must match the dominant language of the text for natural prosody.')
                        vd_instruct = gr.Textbox(label='Voice description', value='体现撒娇稚嫩的萝莉女声，音调偏高且起伏明显', lines=2,
                                                info='Be specific: age, gender, accent, emotion, pitch, pace, breathiness. The model follows detailed instructions well.')
                        vd_btn = gr.Button('Generate (VoiceDesign)', variant='primary')
                    with gr.Column():
                        vd_audio = gr.Audio(label='Output', type='filepath')
                        vd_status = gr.Markdown('')
                vd_btn.click(ui_voice_design, [vd_text, vd_lang, vd_instruct], [vd_audio, vd_status])

        with gr.Tab('Voice Clone'):
            gr.Markdown('**Clone any voice from a 3+ second reference clip.** Provide the transcript for best quality — or enable x-vector-only mode (faster, lower fidelity).')
            with gr.Row():
                with gr.Column():
                    clone_text = gr.Textbox(label='Text to synthesize', value='I am solving the equation x equals negative b plus or minus the square root of b squared minus four a c, all divided by two a.', lines=4,
                                                info='Any text. Will be spoken in the voice of the reference audio.')
                    clone_lang = gr.Dropdown(LANGUAGES, value='English', label='Language',
                                                info='Language of the text to synthesize. The reference audio can be in any supported language.')
                    clone_ref = gr.Audio(label='Reference audio (3+ seconds, clear speech, no background music)', type='filepath',
                                            info='3-30 seconds of clean speech. Longer = better fidelity. No music, no reverb. WAV/FLAC preferred over MP3.')
                    clone_ref_text = gr.Textbox(label='Reference transcript (what the reference audio says)', value='', lines=2,
                                                    info='What the reference audio says. For best results: literal transcript in the same language. Empty = x-vector-only mode (lower fidelity).')
                    clone_xv = gr.Checkbox(label='x-vector only (skip transcript — faster but lower quality)', value=False,
                                             info='Enable if you have no transcript. Quality drops noticeably, especially for non-English.')
                    clone_btn = gr.Button('Generate (Clone)', variant='primary')
                with gr.Column():
                    clone_audio = gr.Audio(label='Output', type='filepath')
                    clone_status = gr.Markdown('')
            clone_btn.click(ui_voice_clone, [clone_text, clone_lang, clone_ref, clone_ref_text, clone_xv], [clone_audio, clone_status])

        with gr.Tab('VRAM'):
            gr.Markdown('**Free a model from VRAM** to load another. Each Qwen3-TTS model uses 3–6 GB.')
            with gr.Row():
                free_cv = gr.Button('Free CustomVoice')
                free_vd = gr.Button('Free VoiceDesign')
                free_base = gr.Button('Free Base (Clone)')
            free_status = gr.Markdown('')
            free_cv.click(lambda: ui_free('CustomVoice'), None, free_status)
            free_vd.click(lambda: ui_free('VoiceDesign'), None, free_status)
            free_base.click(lambda: ui_free('Base'), None, free_status)

    gr.Markdown('**Tips:** Start with the pre-loaded examples. Use 1.7B if you have the VRAM — it sounds noticeably better than 0.6B, especially for non-English languages.')

    gr.Examples(
        label='Quick starts (CustomVoice tab — click a row to load)',
        examples=[
            ['Hello! Welcome to Qwen3-TTS, the latest open-weights text-to-speech model from Alibaba.', 'English', 'Ryan', ''],
            ["Bonjour, je m'appelle Qwen. Bienvenue dans cette démonstration.", 'French', 'Ryan', ''],
            ['こんにちは。Qwen3-TTSのデモンストレーションへようこそ。', 'Japanese', 'Ono_Anna', ''],
            ['The quick brown fox jumps over the lazy dog. Pack my box with five dozen liquor jugs.', 'English', 'Aiden', 'Very fast, with crisp diction.', '', ''],
            ['A young student asked, "Why is the sky blue?" and the teacher smiled before answering.', 'English', 'Serena', 'Gentle, curious, like reading a bedtime story.', '', ''],
        ],
        inputs=[cv_text, cv_lang, cv_speaker, cv_instruct],
    )

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)


demo.load(lambda: "🎙️ Qwen3-TTS ready. Three modes: 9 premium voices, text-prompted voice design, and 3-second cloning.", outputs=[cv_status])


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single TTS call from the notebook. Useful for scripting or quick checks.

MODE = 'custom_voice' # @param ["custom_voice", "voice_design", "voice_clone"] {type:"string"}
TEXT = "Hello! This is Qwen3-TTS running in Google Colab." # @param {type:"string"}
LANGUAGE = "Auto" # @param ["Auto", "Chinese", "English", "Japanese", "Korean", "German", "French", "Russian", "Portuguese", "Spanish", "Italian"]
SPEAKER = "Ryan" # @param ["Vivian", "Serena", "Uncle_Fu", "Dylan", "Eric", "Ryan", "Aiden", "Ono_Anna", "Sohee"]
INSTRUCT = "" # @param {type:"string"}
REF_AUDIO = "" # @param {type:"string"}
REF_TEXT = "" # @param {type:"string"}

if MODE == 'custom_voice':
    path, sr = synth_custom_voice(TEXT, LANGUAGE, SPEAKER, INSTRUCT)
elif MODE == 'voice_design':
    path, sr = synth_voice_design(TEXT, LANGUAGE, INSTRUCT)
else:
    if not REF_AUDIO:
        raise SystemExit('REF_AUDIO is required for voice_clone mode (path to a wav/mp3 file).')
    path, sr = synth_voice_clone(TEXT, LANGUAGE, REF_AUDIO, REF_TEXT, x_vector_only=not REF_TEXT)

print(f'[Done] {path} ({sr} Hz)')
from IPython.display import Audio, display
display(Audio(path))

In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each line in `BATCH` becomes one output file. Useful for audiobooks, narration, voice prototyping.

BATCH_MODE = 'custom_voice' # @param ["custom_voice", "voice_design", "voice_clone"]
BATCH_LANGUAGE = 'English' # @param ["Auto", "Chinese", "English", "Japanese", "Korean", "German", "French", "Russian", "Portuguese", "Spanish", "Italian"]
BATCH_SPEAKER = 'Ryan' # @param ["Vivian", "Serena", "Uncle_Fu", "Dylan", "Eric", "Ryan", "Aiden", "Ono_Anna", "Sohee"]
BATCH_INSTRUCT = '' # @param {type:"string"}
BATCH_REF_AUDIO = '' # @param {type:"string"}
BATCH_REF_TEXT = '' # @param {type:"string"}

BATCH = """\
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
To be, or not to be: that is the question.
It was the best of times, it was the worst of times.
All happy families are alike; each unhappy family is unhappy in its own way.
"""

lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
out_dir = '/content/audio_out/batch'
os.makedirs(out_dir, exist_ok=True)

for i, line in enumerate(lines):
    print(f'[{i+1}/{len(lines)}] {line[:60]}{"..." if len(line) > 60 else ""}')
    try:
        if BATCH_MODE == 'custom_voice':
            path, sr = synth_custom_voice(line, BATCH_LANGUAGE, BATCH_SPEAKER, BATCH_INSTRUCT)
        elif BATCH_MODE == 'voice_design':
            path, sr = synth_voice_design(line, BATCH_LANGUAGE, BATCH_INSTRUCT)
        else:
            path, sr = synth_voice_clone(line, BATCH_LANGUAGE, BATCH_REF_AUDIO, BATCH_REF_TEXT, x_vector_only=not BATCH_REF_TEXT)
        os.rename(path, os.path.join(out_dir, f'{i:03d}.wav'))
    except Exception as e:
        print(f'  -> EXCEPTION: {e}')
        continue

print(f'\n[Done] {len(lines)} files written to {out_dir}')